In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [1]:
%pip install ipynb

Note: you may need to restart the kernel to use updated packages.


In [2]:
# importing data frame from previous notebook

import importlib


d_c = importlib.import_module("ipynb.fs.full.01_DataCleaning")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7200 entries, 0 to 7199
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   account_id                   7200 non-null   object 
 1   account_holder_name          7200 non-null   object 
 2   account_type                 7200 non-null   object 
 3   home_country                 7200 non-null   object 
 4   currency                     7200 non-null   object 
 5   kyc_tier                     7200 non-null   object 
 6   account_created_date         7200 non-null   object 
 7   personal_spend_baseline_usd  7200 non-null   float64
dtypes: float64(1), object(7)
memory usage: 450.1+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 24 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   transaction_id              

In [3]:
d_c.merged_df.shape

(126000, 23)

In [4]:
d_c.merged_df.drop(columns=['account_age_days'],inplace=True)


In [12]:
d_c.merged_df.shape

(126000, 22)

### Feature Engineering

##### Data Re-inspection



In [5]:
d_c.merged_df.describe()

,personal_spend_baseline_usd,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,is_fraud
count,126000.000000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,126000.000000,126000.000000
mean,292.277286,13.542111,7.964918e+05,7.921400,7.675995e+05,0.011429,0.020198,0.030913,0.026603
std,631.504181,5.260369,4.054387e+06,153.566706,3.290377e+06,0.110323,0.140679,0.173082,0.160921
min,3.770000,0.000000,-1.834602e+06,0.000000,-1.669153e+06,0.000000,0.000000,0.000000,0.000000
25%,44.510000,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,0.000000,0.000000
50%,92.190000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,0.000000,0.000000
75%,296.230000,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,0.000000,0.000000
max,10469.650000,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,1.000000,1.000000


##### Fixing negative values in amount columns and avg transaction amount in 30days

In [6]:
# Clip all values below zero to zero

d_c.merged_df["amount"] = d_c.merged_df["amount"].clip(lower=0)
d_c.merged_df["avg_transaction_amount_30d"] = d_c.merged_df["avg_transaction_amount_30d"].clip(lower=0)


In [7]:
d_c.merged_df.describe()

,personal_spend_baseline_usd,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,is_new_device,is_fraud
count,126000.000000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,126000.000000,126000.000000
mean,292.277286,13.542111,7.966235e+05,7.921400,7.676637e+05,0.011429,0.020198,0.030913,0.026603
std,631.504181,5.260369,4.054351e+06,153.566706,3.290356e+06,0.110323,0.140679,0.173082,0.160921
min,3.770000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,44.510000,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,0.000000,0.000000
50%,92.190000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,0.000000,0.000000
75%,296.230000,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,0.000000,0.000000
max,10469.650000,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,1.000000,1.000000


#### Converting time stamp to pd datetime

In [9]:
# Convert timestamp column to datetime
import pandas as pd
d_c.merged_df['timestamp'] = pd.to_datetime(d_c.merged_df['timestamp'],errors='coerce')

In [10]:
d_c.merged_df['year'] = d_c.merged_df['timestamp'].dt.year
d_c.merged_df['month'] = d_c.merged_df['timestamp'].dt.month
d_c.merged_df['day_of_week'] = d_c.merged_df['timestamp'].dt.dayofweek
d_c.merged_df['hour'] = d_c.merged_df['timestamp'].dt.hour


##### converting day and time component into cyclical pattern

In [13]:
import numpy as np
d_c.merged_df['hour_sin'] = np.sin(2 * np.pi * d_c.merged_df['hour']/24)
d_c.merged_df['hour_cos'] = np.cos(2 * np.pi * d_c.merged_df['hour']/24)

d_c.merged_df['dow_sin'] = np.sin(2 * np.pi * d_c.merged_df['day_of_week']/7)
d_c.merged_df['dow_cos'] = np.cos(2 * np.pi * d_c.merged_df['day_of_week']/7)

##### Normalizing Amounts to single currency (USD) using exchange rates at time of transactions

In [14]:
d_c.merged_df.currency_y.value_counts()

currency_y
NGN    64185
USD    24900
GBP    18966
EUR    17949
Name: count, dtype: int64

In [15]:
# Identifying minimum and maximum timestamp
print(d_c.merged_df['timestamp'].min())
print(d_c.merged_df['timestamp'].max())

2025-02-01 00:05:00
2026-08-30 23:33:00


In [16]:
# importing online exchange rate
import requests
start_date = "2025-02-01"
end_date = "2026-08-30"

currencies = ['NGN', 'GBP', 'EUR']
all_rates = []

for currency in currencies:
    url = "https://api.frankfurter.dev/v2/rates"
    params = {
        "from": start_date,
        "to": end_date,
        "base": currency,
        "quotes": "USD"   # correct parameter name is 'symbols', not 'quotes'
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    
    # Flatten the rates dictionary into a DataFrame
    fx_df = pd.DataFrame(data)  # transpose so dates are rows
    
    all_rates.append(fx_df)
    
# Combine all into one DataFrame
fx_conv = pd.concat(all_rates, ignore_index= True)

In [17]:
fx_conv.head()

,date,base,quote,rate
0,2025-01-31,NGN,USD,0.00067
1,2025-02-02,NGN,USD,0.00067
2,2025-02-03,NGN,USD,0.00068
3,2025-02-04,NGN,USD,0.00068
4,2025-02-05,NGN,USD,0.00067


In [18]:
# Ensure column names are consistent
fx_conv = fx_conv.rename(columns={
    'date': 'fx_date',      
    'base': 'currency',     
    'rate': 'rate_to_usd'       
})

# Convert fx_date to datetime
fx_conv['fx_date'] = pd.to_datetime(fx_conv['fx_date'])

# Keep only the relevant columns
fx_conv = fx_conv[['fx_date', 'currency', 'rate_to_usd']]

In [19]:
fx_conv

,fx_date,currency,rate_to_usd
0,2025-01-31,NGN,0.00067
1,2025-02-02,NGN,0.00067
2,2025-02-03,NGN,0.00068
3,2025-02-04,NGN,0.00068
4,2025-02-05,NGN,0.00067
...,...,...,...
1700,2026-08-26,EUR,1.16720
1701,2026-08-27,EUR,1.16550
1702,2026-08-28,EUR,1.16490
1703,2026-08-29,EUR,1.16470


In [20]:
# Create USD baseline rates
usd_rates = pd.DataFrame({
    'fx_date': pd.date_range(
        start='2025-02-01',
        end='2026-08-30',
        freq='D'
    ),
    'currency': 'USD',
    'rate_to_usd': 1.0
})

# Concatenate with existing fx_conv DataFrame
fx_conv = pd.concat(
    [fx_conv, usd_rates],
    ignore_index=True
)

In [21]:
fx_conv

,fx_date,currency,rate_to_usd
0,2025-01-31,NGN,0.00067
1,2025-02-02,NGN,0.00067
2,2025-02-03,NGN,0.00068
3,2025-02-04,NGN,0.00068
4,2025-02-05,NGN,0.00067
...,...,...,...
2276,2026-08-26,USD,1.00000
2277,2026-08-27,USD,1.00000
2278,2026-08-28,USD,1.00000
2279,2026-08-29,USD,1.00000


In [22]:
# Preparing to merge fx table to main data

# Convert timestamp column to datetime
d_c.merged_df['timestamp'] = pd.to_datetime(d_c.merged_df['timestamp'], errors='coerce')

# Normalize timestamp to midnight (removes time component, keeps date)
d_c.merged_df['fx_date'] = d_c.merged_df['timestamp'].dt.normalize()

# Clean currency_y column: strip whitespace and convert to uppercase
d_c.merged_df['currency_y'] = d_c.merged_df['currency_y'].str.strip().str.upper()

In [27]:
# Sort both DataFrames by fx_date 
transactions = d_c.merged_df.sort_values('fx_date')
rates = fx_conv.sort_values('fx_date')

# Perform asof merge:
#    - Match each transaction to the latest available FX rate
#    - Align by currency (currency_y vs currency)
#    - Use backward direction (closest rate on or before transaction date)
d_c.merged_df = pd.merge_asof(
    transactions,
    rates[['fx_date', 'currency', 'rate_to_usd']],
    left_on='fx_date',
    right_on='fx_date',
    left_by='currency_y',
    right_by='currency',
    direction='backward'
)


In [28]:
d_c.merged_df[['timestamp','currency_y','amount','rate_to_usd']].head()

,timestamp,currency_y,amount,rate_to_usd
0,2025-02-01 11:34:00,USD,4362.60,1.00000
1,2025-02-01 07:53:00,NGN,5701952.00,0.00067
2,2025-02-01 09:13:00,NGN,1246983.00,0.00067
3,2025-02-01 06:18:00,USD,71.85,1.00000
4,2025-02-01 18:54:00,NGN,30226.36,0.00067


In [29]:
# Convert amounts to USD using the exchange rate
d_c.merged_df['amount_usd'] = d_c.merged_df['amount'] * d_c.merged_df['rate_to_usd']

# Convert 30-day average transaction amount to USD
d_c.merged_df['avg_transaction_amount_30d_usd'] = (
    d_c.merged_df['avg_transaction_amount_30d'] * d_c.merged_df['rate_to_usd']
)

In [30]:
d_c.merged_df[['timestamp','currency_y','amount','avg_transaction_amount_30d','rate_to_usd',
               'amount_usd','avg_transaction_amount_30d_usd']].head(10)

,timestamp,currency_y,amount,avg_transaction_amount_30d,rate_to_usd,amount_usd,avg_transaction_amount_30d_usd
0,2025-02-01 11:34:00,USD,4362.60,4362.60,1.00000,4362.600000,4362.600000
1,2025-02-01 07:53:00,NGN,5701952.00,6236453.31,0.00067,3820.307840,4178.423718
2,2025-02-01 09:13:00,NGN,1246983.00,1409253.00,0.00067,835.478610,944.199510
3,2025-02-01 06:18:00,USD,71.85,32.93,1.00000,71.850000,32.930000
4,2025-02-01 18:54:00,NGN,30226.36,30226.36,0.00067,20.251661,20.251661
5,2025-02-01 18:53:00,NGN,39444.21,39444.21,0.00067,26.427621,26.427621
6,2025-02-01 14:39:00,GBP,19.00,217.08,1.24320,23.620800,269.873856
7,2025-02-01 18:33:00,GBP,20.72,20.72,1.24320,25.759104,25.759104
8,2025-02-01 21:30:00,NGN,165705.55,165705.55,0.00067,111.022718,111.022718
9,2025-02-01 08:46:00,USD,16.00,16.00,1.00000,16.000000,16.000000


In [32]:
# Generating more Behavioural features

# Create a new column comparing amount_usd to personal baseline
d_c.merged_df['amount_vs_personal_baseline'] = (
    d_c.merged_df['amount_usd'] /
    d_c.merged_df['personal_spend_baseline_usd'].replace(0, np.nan)
)

# Calculate deviation from 30-day average transaction amount in USD
d_c.merged_df['amount_deviation_30d_usd'] = (
    d_c.merged_df['amount_usd'] -
    d_c.merged_df['avg_transaction_amount_30d_usd']
)

# Flag anomalies: transactions from new devices with unusual amount ratios
d_c.merged_df['new_device_amount_anomaly'] = (
    d_c.merged_df['is_new_device'].fillna(0) *
    d_c.merged_df['amount_to_avg_ratio']
)

# Creating Account Age in days

#Convert both columns to datetime
d_c.merged_df['account_created_date'] = pd.to_datetime(
    d_c.merged_df['account_created_date'],
    errors='coerce'
)

d_c.merged_df['timestamp'] = pd.to_datetime(
    d_c.merged_df['timestamp'],
    errors='coerce'
)

# Account age in days at transaction time
d_c.merged_df['account_age_days'] = (
    d_c.merged_df['timestamp'] - d_c.merged_df['account_created_date']
).dt.days


In [33]:
d_c.merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126000 entries, 0 to 125999
Data columns (total 38 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   account_created_date            126000 non-null  datetime64[ns]
 1   personal_spend_baseline_usd     126000 non-null  float64       
 2   account_type_y                  126000 non-null  object        
 3   kyc_tier_y                      126000 non-null  object        
 4   timestamp                       126000 non-null  datetime64[ns]
 5   day_of_week                     126000 non-null  int32         
 6   hour_of_day                     126000 non-null  int64         
 7   description                     126000 non-null  object        
 8   merchant_name                   126000 non-null  object        
 9   merchant_category               126000 non-null  object        
 10  channel                         126000 non-null  object 

In [34]:
d_c.merged_df.describe()

,account_created_date,personal_spend_baseline_usd,timestamp,day_of_week,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,...,dow_sin,dow_cos,fx_date,rate_to_usd,amount_usd,avg_transaction_amount_30d_usd,amount_vs_personal_baseline,amount_deviation_30d_usd,new_device_amount_anomaly,account_age_days
count,126000,126000.000000,126000,126000.000000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,...,126000.000000,126000.000000,126000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000
mean,2025-01-02 20:52:42.514285824,292.277286,2025-11-17 18:16:58.262857216,3.016540,13.542111,7.966235e+05,7.921400,7.676637e+05,0.011429,0.020198,...,-0.005871,0.003495,2025-11-17 04:14:51.428570880,0.562762,1094.204081,1167.218627,2.382768,-73.014546,0.237861,318.307048
min,2021-12-25 00:00:00,3.770000,2025-02-01 00:05:00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,...,-0.974928,-0.900969,2025-02-01 00:00:00,0.000610,0.000000,0.000000,0.000000,-134344.210938,0.000000,-560.000000
25%,2024-05-15 00:00:00,44.510000,2025-07-11 09:22:00,1.000000,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,...,-0.781831,-0.900969,2025-07-11 00:00:00,0.000690,20.303220,28.740000,0.336148,-113.358355,0.000000,35.000000
50%,2025-02-14 00:00:00,92.190000,2025-11-27 22:32:30,3.000000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,...,0.000000,-0.222521,2025-11-27 00:00:00,0.000740,73.789277,101.040150,0.766933,-0.810000,0.000000,289.000000
75%,2025-10-07 00:00:00,296.230000,2026-03-24 12:33:30,5.000000,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,...,0.781831,0.623490,2026-03-24 00:00:00,1.151400,442.712662,732.009048,2.339176,35.941814,0.000000,577.000000
max,2026-08-29 00:00:00,10469.650000,2026-08-30 23:33:00,6.000000,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,...,0.974928,1.000000,2026-08-30 00:00:00,1.381000,298224.151200,138394.295438,73.206867,279836.639820,3107.800000,1682.000000
std,NaN,631.504181,NaN,2.006247,5.260369,4.054351e+06,153.566706,3.290356e+06,0.110323,0.140679,...,0.708902,0.705280,NaN,0.581247,4511.626342,3499.380210,3.942715,4094.325991,16.288476,378.997935


##### Handling negative value in account_age_days

In [36]:
d_c.merged_df["account_age_days"] = d_c.merged_df["account_age_days"].clip(lower=0)

In [37]:
d_c.merged_df.describe()

,account_created_date,personal_spend_baseline_usd,timestamp,day_of_week,hour_of_day,amount,amount_to_avg_ratio,avg_transaction_amount_30d,transaction_velocity_1h,is_cross_border,...,dow_sin,dow_cos,fx_date,rate_to_usd,amount_usd,avg_transaction_amount_30d_usd,amount_vs_personal_baseline,amount_deviation_30d_usd,new_device_amount_anomaly,account_age_days
count,126000,126000.000000,126000,126000.000000,126000.000000,1.260000e+05,126000.000000,1.260000e+05,126000.000000,126000.000000,...,126000.000000,126000.000000,126000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000,126000.000000
mean,2025-01-02 20:52:42.514285824,292.277286,2025-11-17 18:16:58.262857216,3.016540,13.542111,7.966235e+05,7.921400,7.676637e+05,0.011429,0.020198,...,-0.005871,0.003495,2025-11-17 04:14:51.428570880,0.562762,1094.204081,1167.218627,2.382768,-73.014546,0.237861,354.721992
min,2021-12-25 00:00:00,3.770000,2025-02-01 00:05:00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00,0.000000,0.000000,...,-0.974928,-0.900969,2025-02-01 00:00:00,0.000610,0.000000,0.000000,0.000000,-134344.210938,0.000000,0.000000
25%,2024-05-15 00:00:00,44.510000,2025-07-11 09:22:00,1.000000,9.000000,6.310500e+01,0.250000,9.242750e+01,0.000000,0.000000,...,-0.781831,-0.900969,2025-07-11 00:00:00,0.000690,20.303220,28.740000,0.336148,-113.358355,0.000000,35.000000
50%,2025-02-14 00:00:00,92.190000,2025-11-27 22:32:30,3.000000,14.000000,5.942240e+03,0.980000,5.942240e+03,0.000000,0.000000,...,0.000000,-0.222521,2025-11-27 00:00:00,0.000740,73.789277,101.040150,0.766933,-0.810000,0.000000,289.000000
75%,2025-10-07 00:00:00,296.230000,2026-03-24 12:33:30,5.000000,18.000000,1.208470e+05,1.830000,1.575701e+05,0.000000,0.000000,...,0.781831,0.623490,2026-03-24 00:00:00,1.151400,442.712662,732.009048,2.339176,35.941814,0.000000,577.000000
max,2026-08-29 00:00:00,10469.650000,2026-08-30 23:33:00,6.000000,23.000000,2.691614e+08,31762.690000,1.733621e+08,3.000000,1.000000,...,0.974928,1.000000,2026-08-30 00:00:00,1.381000,298224.151200,138394.295438,73.206867,279836.639820,3107.800000,1682.000000
std,NaN,631.504181,NaN,2.006247,5.260369,4.054351e+06,153.566706,3.290356e+06,0.110323,0.140679,...,0.708902,0.705280,NaN,0.581247,4511.626342,3499.380210,3.942715,4094.325991,16.288476,331.327033
